**S — Single Responsibility Principle (SRP)** <br>

Definition: "A class should have only **ONE** reason to change."

Not: <br>
- One method
- One feature

But: <br>
- One Responsibility
- One axis of change.

Why SRP matters in backend systems <br>
Without **SRP** <br>
- Code becomes fragile
- Small change break unrelated features
- Testing becomes painful
- Scaling the system is hard


**BAD DESIGN (Violates SRP)** <br>

Let's say you're building a User Registration flow.

In [ ]:
class UserService:
    def register(self, user_data):
        # 1. Validate data
        if "email" not in user_data or "password" not in user_data:
            raise ValueError("Email & Password are mandatory")
        # 2. Save data to database except password
        print("User saved to DB")
        # 3. Hash password and update entity
        print("Generated hash of password and update entity")
        # 4. Send welcome email
        print("Email sent successfully")
        # 5. Log activity (YYYY-MM-DD: abc@example registered successfully)
        print("Log is recorded")

**Why this is bad** <br>

This class has MULTIPLE responsibilities:

| Responsibility | Reason to Change        |
| -------------- | ----------------------- |
| Validation     | Validation rules change |
| Hashing        | Using new hashing algo  |
| Persistence    | DB changes              |
| Notification   | Email provider changes  |
| Logging        | Logging system changes  |

👉 5 reasons to change = SRP violation


**GOOD DESIGN (Follows SRP)**

We split responsibilities and use design patterns properly.

**1. Validation (Strategy Pattern)**

Different validation rules = different strategies

😎 Remember: <br>
- Strategy Pattern means: A same destination but different strategies.

In [2]:
# interface
class ValidationStrategy:
    def validate(self, data):
        raise NotImplementedError

class UserValidationStrategy(ValidationStrategy):
    def validate(self, data):
        if "email" not in data:
            raise ValueError("Email and Password are mandatory")
        
        if "password" not in data:
            raise ValueError("Email and Password are mandatory")

        if len(data.get("password", "")) < 8:
            raise ValueError("Password must be at least 8 characters")

✔ Single responsibility: validate user data

**2/3. Save Data to DB (Repository/Facade)**

Data save in database and password hashing involve many steps and complexity. So, we need a single interface. The complexity will be handled by someone else.

A system is too complex:<br>
* Many classes  <br>
* Many steps <br>
* Hard to use correctly.

You want:
- **One Simple Interface**

Solution: **Facade** Design Pattern

In [28]:
import uuid
from abc import abstractmethod

class User:
    def __init__(self, username, email):
        self.username = username
        self.email = email

# Implement MyHashed class by following facade pattern to hide
# implement complexity and ensure single responsibility
class MyHashed:
    @abstractmethod
    def hash(password):
        return str(uuid.uuid5(uuid.NAMESPACE_DNS, password))

class UserFacade:
    def save(self, user_data):
        print(user_data)
        user = User(user_data.get("email"), user_data.get("password"))
        hashed_password = MyHashed.hash(user_data.get("password")) 
        user.hashed_password = hashed_password
        print("User registered successfully")
        return user

✔ Single responsibility: Database related change.

**4. Notification (Observer Pattern)**

You have:<br>
- One object (Subject) whose state changes. In this case, registration
- Many dependent objects (Observers) that must react. In this case, email service.

But: <br>
- You don't want tight coupling
- You don't want to hardcode calls everywhere

Solution: **Observer** pattern.

In [5]:
class ObserverInterface:
    def update(self, data):
        raise NotImplementedError
    

class EmailObserver(ObserverInterface):
    def update(self, data):
        print("Welcome email sent")

✔ Single responsibility: react to events

👉 Do you notice one thing? Notification service can be multiple right? Email, SMS etc! That means many strategies/ways for a single goal. Which pattern? Strategy! Yes! You're right. That's why Observer and Strategy are in Behavior category!

**4. Logging (Decorator Pattern)**

When you need to add extra behavior without modifing existing code then decorator pattern is the solution e.g. Logging, Auth

In [38]:
class LoggerDecorator:
    def __init__(self, service):
        self.service = service 
    
    def execute(self, data):
        service_name = self.service.__class__.__name__
        print(f"Action started: {service_name} with data: {data}")
        self.service.execute(data)

✔ Responsibility: logging only

**6. Orchestration (Facade Pattern)**

This class coordinates, not executes logic.

In [31]:
class UserRegistrationFacade:
    def __init__(self, validator, user_db, observers):
        self.validator = validator
        self.user_db = user_db
        self.observers = observers
    
    def execute(self, user_data):
        self.validator.validate(user_data)
        response = self.user_db.save(user_data)
        print(response.__dict__)

        for observer in self.observers:
            observer.update(user_data)

✔ Facade responsibility: coordinate the registration flow

**Usage**

In [39]:
validator = UserValidationStrategy()
user_db = UserFacade()
observers = [EmailObserver()]

# create facade
registration_facade = UserRegistrationFacade(
    validator=validator,
    user_db=user_db,
    observers=observers
)

# Decorate facade with logging
logged_registration = LoggerDecorator(registration_facade)


user_data = {
    "email": "shamim@example.com",
    "password": "123456789",
    "name": "Shamim"
}

logged_registration.execute(user_data)

Action started: UserRegistrationFacade with data: {'email': 'shamim@example.com', 'password': '123456789', 'name': 'Shamim'}
{'email': 'shamim@example.com', 'password': '123456789', 'name': 'Shamim'}
User registered successfully
{'username': 'shamim@example.com', 'email': '123456789', 'hashed_password': '2fc772f7-e2b3-5d9c-bd58-6153578f4c6c'}
Welcome email sent


See every class has only **ONE** responsibility.